In [1]:
import pandas as pd
import duckdb
import polars as pl
import time

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
df = pd.read_csv("teen_phone_addiction_dataset.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (3000, 25)


,ID,Name,Age,Gender,Location,School_Grade,Daily_Usage_Hours,Sleep_Hours,Academic_Performance,Social_Interactions,...,Screen_Time_Before_Bed,Phone_Checks_Per_Day,Apps_Used_Daily,Time_on_Social_Media,Time_on_Gaming,Time_on_Education,Phone_Usage_Purpose,Family_Communication,Weekend_Usage_Hours,Addiction_Level
0,1,Shannon Francis,13,Female,Hansonfort,9th,4.0,6.1,78,5,...,1.4,86,19,3.6,1.7,1.2,Browsing,4,8.7,10.0
1,2,Scott Rodriguez,17,Female,Theodorefort,7th,5.5,6.5,70,5,...,0.9,96,9,1.1,4.0,1.8,Browsing,2,5.3,10.0
2,3,Adrian Knox,13,Other,Lindseystad,11th,5.8,5.5,93,8,...,0.5,137,8,0.3,1.5,0.4,Education,6,5.7,9.2
3,4,Brittany Hamilton,18,Female,West Anthony,12th,3.1,3.9,78,8,...,1.4,128,7,3.1,1.6,0.8,Social Media,8,3.0,9.8
4,5,Steven Smith,14,Other,Port Lindsaystad,9th,2.5,6.7,56,4,...,1.0,96,20,2.6,0.9,1.1,Gaming,10,3.7,8.6


In [3]:
con = duckdb.connect()

con.register("students", df)

print("Dataset loaded into DuckDB successfully!")

Dataset loaded into DuckDB successfully!


In [4]:
query1 = con.execute("""
SELECT COUNT(*) AS Total_Records
FROM students
""").fetchdf()

query1

,Total_Records
0,3000


In [5]:
query2 = con.execute("""
SELECT AVG(Daily_Usage_Hours) AS Avg_Daily_Usage
FROM students
""").fetchdf()

query2

,Avg_Daily_Usage
0,5.020667


In [6]:
query3 = con.execute("""
SELECT AVG(Academic_Performance) AS Avg_Academic_Performance
FROM students
""").fetchdf()

query3

,Avg_Academic_Performance
0,74.947333


In [7]:
query4 = con.execute("""
SELECT *
FROM students
ORDER BY Daily_Usage_Hours DESC
LIMIT 10
""").fetchdf()

query4

,ID,Name,Age,Gender,Location,School_Grade,Daily_Usage_Hours,Sleep_Hours,Academic_Performance,Social_Interactions,...,Screen_Time_Before_Bed,Phone_Checks_Per_Day,Apps_Used_Daily,Time_on_Social_Media,Time_on_Gaming,Time_on_Education,Phone_Usage_Purpose,Family_Communication,Weekend_Usage_Hours,Addiction_Level
0,2671,Deborah Gonzalez,16,Other,North Julieland,8th,11.5,8.6,96,0,...,0.2,58,12,3.0,0.0,1.4,Browsing,5,4.8,10.0
1,2401,Lisa Gonzalez,13,Male,Deborahmouth,11th,11.2,8.1,55,1,...,0.9,34,17,3.6,2.4,0.9,Browsing,5,8.2,10.0
2,711,Pamela Johnson,15,Male,Derrickville,8th,11.0,7.7,59,2,...,2.1,81,10,1.6,1.6,1.4,Browsing,8,6.2,10.0
3,2747,Miranda Lee,15,Male,North Henryfort,7th,11.0,8.1,75,8,...,0.3,139,19,3.2,4.0,1.2,Social Media,5,4.1,10.0
4,1666,Brian Pierce,15,Other,Fordtown,9th,10.9,5.2,54,1,...,1.2,68,16,3.2,0.9,1.9,Other,5,6.1,10.0
5,1732,Patrick Adams,14,Female,Walkerview,8th,10.6,5.1,94,6,...,0.3,77,10,1.7,0.0,0.4,Browsing,7,3.5,10.0
6,228,Michael Hopkins,18,Female,North Austin,9th,10.6,9.1,93,8,...,1.4,36,7,3.7,1.2,2.5,Other,8,6.3,10.0
7,882,Bridget Santos,18,Other,South Bobbyland,10th,10.6,6.7,79,6,...,0.5,97,20,1.0,1.8,2.2,Education,1,7.4,10.0
8,2751,Jamie Salas,15,Female,North Jacobstad,7th,10.6,7.5,81,9,...,0.0,39,12,1.1,2.4,0.5,Other,7,2.5,10.0
9,583,Julia Rogers,16,Female,Desireeland,8th,10.5,8.0,97,10,...,1.0,128,16,1.9,0.0,0.9,Social Media,10,3.7,10.0


In [8]:
query5 = con.execute("""
SELECT Gender,
AVG(Daily_Usage_Hours) AS Avg_Usage
FROM students
GROUP BY Gender
""").fetchdf()

query5

,Gender,Avg_Usage
0,Other,4.952508
1,Female,5.052532
2,Male,5.054626


In [9]:
import time

# Pandas timing
start = time.time()
pandas_result = df["Daily_Usage_Hours"].mean()
pandas_time = time.time() - start

# DuckDB timing
start = time.time()
duckdb_result = con.execute("""
SELECT AVG(Daily_Usage_Hours)
FROM students
""").fetchone()[0]
duckdb_time = time.time() - start

print(f"Pandas Time: {pandas_time:.6f} seconds")
print(f"DuckDB Time: {duckdb_time:.6f} seconds")

Pandas Time: 0.004031 seconds
DuckDB Time: 0.009122 seconds


In [10]:
pl_df = pl.read_csv("teen_phone_addiction_dataset.csv")

print(pl_df.head())

shape: (5, 25)
┌─────┬──────────────┬─────┬────────┬───┬──────────────┬──────────────┬──────────────┬─────────────┐
│ ID  ┆ Name         ┆ Age ┆ Gender ┆ … ┆ Phone_Usage_ ┆ Family_Commu ┆ Weekend_Usag ┆ Addiction_L │
│ --- ┆ ---          ┆ --- ┆ ---    ┆   ┆ Purpose      ┆ nication     ┆ e_Hours      ┆ evel        │
│ i64 ┆ str          ┆ i64 ┆ str    ┆   ┆ ---          ┆ ---          ┆ ---          ┆ ---         │
│     ┆              ┆     ┆        ┆   ┆ str          ┆ i64          ┆ f64          ┆ f64         │
╞═════╪══════════════╪═════╪════════╪═══╪══════════════╪══════════════╪══════════════╪═════════════╡
│ 1   ┆ Shannon      ┆ 13  ┆ Female ┆ … ┆ Browsing     ┆ 4            ┆ 8.7          ┆ 10.0        │
│     ┆ Francis      ┆     ┆        ┆   ┆              ┆              ┆              ┆             │
│ 2   ┆ Scott        ┆ 17  ┆ Female ┆ … ┆ Browsing     ┆ 2            ┆ 5.3          ┆ 10.0        │
│     ┆ Rodriguez    ┆     ┆        ┆   ┆              ┆              ┆     

In [11]:
polars_avg = pl_df.select(
    pl.col("Daily_Usage_Hours").mean()
)

print(polars_avg)

shape: (1, 1)
┌───────────────────┐
│ Daily_Usage_Hours │
│ ---               │
│ f64               │
╞═══════════════════╡
│ 5.020667          │
└───────────────────┘
